In [35]:
import matplotlib
matplotlib.use('Agg')  # 设置为不显示的后端
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
import seaborn as sns
import anndata as ad
import scanpy as sc
import geopandas as gpd
import os
os.chdir("E:/202411_HPV原位测序毕设数据/2_Epi8问题二/2410817///////////////")
os.getcwd()

'E:\\202411_HPV原位测序毕设数据\\2_Epi8问题二\\2410817'

In [36]:
### Epi8分布和单基因表达图

### 结果读取
adata = sc.read_10x_h5('outs/cell_feature_matrix.h5')
mx = adata.to_df()
centroids = pd.read_parquet('outs/cells.parquet', engine='pyarrow')
centroids = centroids.set_index('cell_id')

### 挑取上皮
epi = mx[(mx['KRT5'] > 0) | (mx['KRT13'] > 0)]
centroids = centroids.loc[mx.index, :]
centroids.loc[:,'class'] = 'other'
centroids.loc[epi.index,'class'] = 'epithelium'
centroids['name'] = centroids.index

### 定义epi8
epi8 = epi.loc[:, ['CRYAB', 'GAS5', 'HSPA1B', 'NME2', 'RPL13', 'RPLP1', 'SNHG29', 'SNHG5', 'SNHG8']]
epi8_sum = epi8.sum(axis = 1)
thresholds = epi8_sum.quantile(0.6)
epi8_sum = epi8_sum[epi8_sum >= thresholds]
centroids.loc[epi8_sum.index, 'class'] = 'epi8'

### 砖块图准备
geojson_file = "outs/Cells.geojson"
gdf = gpd.read_file(geojson_file)
gdf = gdf.merge(centroids[['x_centroid', 'y_centroid', 'class', 'name']], on = ['name'], how = 'left')
gdf.dropna(inplace = True)
class_colors = {'epi8': '#df2020', 'epithelium': '#54adc4', 'other': '#f0e48f'}
gdf['color'] = gdf['class'].map(class_colors).fillna('#808080')

### 砖块图可视化
plt.style.use('dark_background')
figure_size = (10,10)
fig, axes = plt.subplots(1,1,figsize = figure_size)
gdf.plot(ax = axes, color = gdf['color'], aspect=1)
axes.invert_yaxis()
# axes.set_aspect('auto')
axes.set_axis_off()
plt.savefig('CellMap.png', dpi = 300, transparent = False, bbox_inches = 'tight')

### 单基因可视化 
adata.obsm['spatial'] = centroids[["x_centroid","y_centroid"]].copy().to_numpy()
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
figure_size = [s * 0.6 for s in figure_size]
for i in mx.columns.to_list():
    plt.rcParams["figure.figsize"] = figure_size
    sc.pl.spatial(adata, color = i, spot_size = 60, show = False, cmap = 'Spectral_r', title = '', colorbar_loc=None)
    plt.gca().set_axis_off()
    plt.savefig(i + '.png', dpi = 300, transparent = False, bbox_inches = 'tight')
    
### 统计平均信号数
mx.mean().sum()

11.38092817567224

In [18]:
### 调整CellMap大小

plt.style.use('dark_background')
figure_size = (10,10)
fig, axes = plt.subplots(1,1,figsize = figure_size)
gdf.plot(ax = axes, color = gdf['color'], aspect=1)
axes.invert_yaxis()
axes.set_aspect('auto')
axes.set_axis_off()
plt.savefig('CellMap.png', dpi = 300, transparent = False, bbox_inches = 'tight')